# SSL4PR Baseline for Parkinson's Disease Classification
## Self-supervised speech representations on the PC-GITA vowels

Baseline following La Quatra et al., *"Exploiting Foundation Models and Speech Enhancement for Parkinson's Disease Detection from Speech in Real-World Operative Conditions"* (Interspeech 2024) — [K-STMLab/SSL4PR](https://github.com/K-STMLab/SSL4PR).

**Model.** WavLM-Base → learnable softmax-weighted sum over the transformer layers → temporal pooling (attention, mean or max) → dropout → linear output trained with `BCEWithLogitsLoss`. The convolutional feature encoder is frozen.

**Data.** All five sustained vowels are pooled into one dataset, read from:
```
DATA_ROOT/Patologicas/{vowel}/AVPEPUDEA0001a1.wav    (PD, label = 1)
DATA_ROOT/Control/{vowel}/AVPEPUDEAC0001a1.wav       (HC, label = 0)
```
The speaker ID is the file name without its last two characters.

**Evaluation.** The same predefined speaker-independent folds as the eaQHM and eGeMAPS notebooks (`master_cv_folds_vowels.csv`, 5 repeats × 10 outer folds). Within each outer fold, learning rate, weight decay, dropout and pooling are tuned with Optuna (20 trials, 5-fold speaker-independent inner CV, ROC AUC). The final model is refitted on the full outer-training set for the mean best number of epochs. Results are reported at sample and speaker level.

### Imports and device

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import os, gc, copy, random, warnings, hashlib, sys
from pathlib import Path
from typing import List, Tuple

os.environ.setdefault("PYTHONHASHSEED", "42")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import optuna
from torch.utils.data import Dataset, DataLoader
from transformers import AutoFeatureExtractor, WavLMModel, HubertModel
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
)
from tqdm import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""))

c:\Users\panat\anaconda3\envs\nedipm10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda  (NVIDIA GeForce RTX 4070 Ti)


### Mixed precision and TF32

In [2]:
# Mixed precision + TF32 (speed)
USE_AMP   = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
if USE_AMP:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    print(f"AMP: enabled | dtype={'bfloat16' if AMP_DTYPE is torch.bfloat16 else 'float16'} | TF32 on")

AMP: enabled | dtype=bfloat16 | TF32 on


### Configuration

In [7]:
# ============================================================
# CONFIG
# ============================================================
DATA_ROOT      = Path(r"D:\PC-GITA_per_task_44100Hz\Vowels")
VOWELS         = ["a", "e", "i", "o", "u"]

BACKBONE_NAME  = "microsoft/wavlm-base"
BACKBONE_TYPE  = "wavlm"
TARGET_SR      = 16_000

TASK            = "vowels"
CSV_PATH        = f"../folds/master_cv_folds_{TASK}.csv"
RESULT_TAG      = "wavlm"

N_REPEATS       = 5
N_OUTER_SPLITS  = 10
N_INNER_SPLITS  = 5
N_OPTUNA_TRIALS = 20
BASE_SEED       = 42

MAX_AUDIO_SEC_OVERRIDE = None

### Logging to file (tee)

In [8]:
# ============================================================
# TEE LOGGING
# ============================================================
class Tee:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log      = open(filepath, "w", buffering=1, encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.terminal.flush()
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def __getattr__(self, name):
        return getattr(self.terminal, name)

sys.stdout = Tee("ssl4pr_output.log")

### 1. Data-loading helpers (folder scan and audio caching)

In [9]:
# ============================================================
# 1. BUILD DATAFRAME FROM FOLDERS
# ============================================================
def parse_speaker_id(stem: str) -> str:
    """AVPEPUDEA0001a1 → AVPEPUDEA0001 ; AVPEPUDEAC0001a1 → AVPEPUDEAC0001"""
    return stem[:-2]


def build_dataframe(data_root: Path, vowels: List[str]) -> pd.DataFrame:
    records = []
    for label_int, class_folder in [(1, "Patologicas"), (0, "Control")]:
        for vowel in vowels:
            folder = data_root / class_folder / vowel
            if not folder.exists():
                print(f"  [WARNING] folder not found: {folder}")
                continue
            for wav_file in sorted(folder.glob("*.wav")):
                stem       = wav_file.stem
                speaker_id = parse_speaker_id(stem)
                records.append({
                    "audio_path": str(wav_file),
                    "label":      label_int,
                    "speaker":    speaker_id,
                    "vowel":      vowel,
                    "filename":   stem,
                })
    return pd.DataFrame(records).reset_index(drop=True)


def preprocess_and_cache(paths, feature_extractor, max_sec, target_sr, cache_dir="audio_cache"):
    """Cache pre-processed feature-extractor inputs as .npy files."""
    Path(cache_dir).mkdir(exist_ok=True)
    cached = []
    for path in paths:
        h          = hashlib.md5(path.encode()).hexdigest()
        cache_path = f"{cache_dir}/{h}.npy"
        if not Path(cache_path).exists():
            wav, sr = torchaudio.load(path)
            if wav.shape[0] > 1:
                wav = wav.mean(0, keepdim=True)
            if sr != target_sr:
                wav = torchaudio.functional.resample(wav, sr, target_sr)
            wav = wav.squeeze(0).numpy()
            max_len = int(max_sec * target_sr)
            if len(wav) >= max_len:
                wav = wav[:max_len]
            else:
                wav = np.pad(wav, (0, max_len - len(wav)))
            iv = feature_extractor(
                wav, sampling_rate=target_sr,
                return_tensors="pt", padding=False
            )["input_values"].squeeze(0)
            np.save(cache_path, iv.numpy())
        cached.append(cache_path)
    return cached

### 2. Load data, gender metadata and folds

In [10]:
# ============================================================
# 2. LOAD DATA + FOLD MAP
# ============================================================
print("Scanning audio folders …")
data = build_dataframe(DATA_ROOT, VOWELS)

# Gender metadata
gender_df = pd.read_csv("../gender_metadata/genders_pc_gita.csv")
gender_df["gender"] = gender_df["gender"].str.strip('"').str.strip()
data = data.merge(gender_df, on="speaker", how="left")

missing_gender = data["gender"].isna().sum()
if missing_gender > 0:
    print(f"[WARNING] {missing_gender} rows missing gender — using 'unknown'")
    data["gender"] = data["gender"].fillna("unknown")

print(f"\nGender × Class:\n"
      f"{data.drop_duplicates('speaker').groupby(['label','gender']).size().unstack(fill_value=0)}")
print(f"\n  Files: {len(data)} | "
      f"Pathological: {data['label'].sum()} | Control: {(data['label']==0).sum()} | "
      f"Speakers: {data['speaker'].nunique()}")
print(data.groupby(["vowel","label"]).size().unstack(fill_value=0).to_string())

# Load and merge fold map
fold_map_raw = pd.read_csv(CSV_PATH)
repeat_cols  = [c for c in fold_map_raw.columns if c.startswith("Repeat_")]

fold_map_spk = (
    fold_map_raw[["speaker"] + repeat_cols]
    .drop_duplicates(subset="speaker")
    .reset_index(drop=True)
)
assert len(fold_map_spk) == fold_map_raw["speaker"].nunique(), \
    "Duplicate speakers with DIFFERENT fold values detected!"

data = data.merge(fold_map_spk, on="speaker", how="left").reset_index(drop=True)
assert data[repeat_cols[0]].isna().sum() == 0, \
    "Some rows have no fold assignment — speaker IDs don't match between folders and CSV."

print(f"\nFold map merged: {CSV_PATH}")
print(f"  {len(data)} files | {data['speaker'].nunique()} speakers")

# Verify fold consistency per speaker
for col in repeat_cols:
    assert data.groupby('speaker')[col].nunique().max() == 1, \
        f"Speaker has inconsistent fold assignments in {col}"
print("✅ All speakers have consistent fold assignments")

data["stratify_key"] = data["label"].astype(str) + "_" + data["gender"]
print(f"\nStratify keys (speaker-level):")
print(data.drop_duplicates("speaker")["stratify_key"].value_counts().sort_index())

# Build aligned arrays
audio_paths = data["audio_path"].tolist()
y           = data["label"].values
y_stratify  = data["stratify_key"].values
groups      = data["speaker"].values

Scanning audio folders …

Gender × Class:
gender  female  male
label               
0           25    25
1           25    25

  Files: 1500 | Pathological: 750 | Control: 750 | Speakers: 100
label    0    1
vowel          
a      150  150
e      150  150
i      150  150
o      150  150
u      150  150

Fold map merged: ../folds/master_cv_folds_vowels.csv
  1500 files | 100 speakers
✅ All speakers have consistent fold assignments

Stratify keys (speaker-level):
stratify_key
0_female    25
0_male      25
1_female    25
1_male      25
Name: count, dtype: int64


### 3. Audio duration analysis and truncation cap

In [11]:
# ============================================================
# 3. AUDIO DURATION ANALYSIS
# ============================================================
print("\nAnalysing audio durations …")
durations = []
for path in audio_paths:
    wav, sr = torchaudio.load(path)
    durations.append(wav.shape[1] / sr)

dur_values = np.array(durations)
print(f"  Max: {dur_values.max():.2f}s | Min: {dur_values.min():.2f}s | "
      f"Mean: {dur_values.mean():.2f}s | Median: {np.median(dur_values):.2f}s")
print(f"  P90: {np.percentile(dur_values,90):.2f}s | "
      f"P95: {np.percentile(dur_values,95):.2f}s | "
      f"P99: {np.percentile(dur_values,99):.2f}s")

if MAX_AUDIO_SEC_OVERRIDE is not None:
    MAX_AUDIO_SEC = int(MAX_AUDIO_SEC_OVERRIDE)
    print(f"  MAX_AUDIO_SEC = {MAX_AUDIO_SEC}s (from MAX_AUDIO_SEC_OVERRIDE — "
          f"independent of test data)")
else:
    MAX_AUDIO_SEC = int(np.ceil(np.percentile(dur_values, 95)))
    print(f" MAX_AUDIO_SEC computed as P95 over the FULL dataset ")
n_truncated = int(np.sum(dur_values > MAX_AUDIO_SEC))
print(f"  MAX_AUDIO_SEC = {MAX_AUDIO_SEC}s | Files truncated: {n_truncated}/{len(dur_values)}")


Analysing audio durations …
  Max: 24.79s | Min: 0.37s | Mean: 2.75s | Median: 2.06s
  P90: 5.29s | P95: 6.86s | P99: 12.94s
 MAX_AUDIO_SEC computed as P95 over the FULL dataset 
  MAX_AUDIO_SEC = 7s | Files truncated: 69/1500


### 4. Utilities

In [12]:
# ============================================================
# 4. UTILITIES
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False


def safe_auc(y_true, y_score) -> float:
    """roc_auc_score that returns NaN instead of raising on single-class input."""
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return roc_auc_score(y_true, y_score)


def aggregate_mean_by_group(
    y: np.ndarray, p: np.ndarray, g: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Average sample-level probabilities per speaker."""
    y, p, g = np.asarray(y).astype(int), np.asarray(p).astype(float), np.asarray(g)
    uniq = np.unique(g)
    y_g  = np.zeros(len(uniq), dtype=int)
    p_g  = np.zeros(len(uniq))
    for i, gg in enumerate(uniq):
        idx    = np.where(g == gg)[0]
        p_g[i] = np.mean(p[idx])
        y_g[i] = int(np.mean(y[idx]) >= 0.5)
    return y_g, p_g, uniq

### 4b. Backbone caching (deepcopy + RNG-state replay)

In [13]:
# ============================================================
# 4b. BACKBONE CACHING  (deepcopy + RNG-state replay)
# ============================================================
_BACKBONE_TEMPLATE  = None   # CPU copy of the pretrained backbone
_RNG_AFTER_BACKBONE = {}     # random_state -> (torch_cpu, py, np) RNG snapshot


def _load_backbone(backbone_name, backbone_type):
    if backbone_type == "wavlm":
        return WavLMModel.from_pretrained(backbone_name, output_hidden_states=True)
    return HubertModel.from_pretrained(backbone_name, output_hidden_states=True)


def _get_backbone_template(backbone_name, backbone_type):
    global _BACKBONE_TEMPLATE
    if _BACKBONE_TEMPLATE is None:
        _BACKBONE_TEMPLATE = _load_backbone(backbone_name, backbone_type)
    return _BACKBONE_TEMPLATE


def _restore_rng_after_backbone(random_state, backbone_name, backbone_type):
    """Put torch(CPU)/numpy/python RNG exactly where the ORIGINAL __init__ would
    be right after from_pretrained, so the head/pool init below is bit-identical.
    CUDA RNG is intentionally untouched: from_pretrained builds on CPU and does
    not consume CUDA RNG, so set_seed()'s CUDA state already matches the original.
    """
    if random_state not in _RNG_AFTER_BACKBONE:
        set_seed(int(random_state))                          # R0 — same start as fit()
        _ = _load_backbone(backbone_name, backbone_type)     # advance RNG as the old path did
        _RNG_AFTER_BACKBONE[random_state] = (
            torch.get_rng_state(), random.getstate(), np.random.get_state(),
        )
        del _
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    t_state, py_state, np_state = _RNG_AFTER_BACKBONE[random_state]
    torch.set_rng_state(t_state)
    random.setstate(py_state)
    np.random.set_state(np_state)

def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total     = trainable + frozen
    return trainable, frozen, total

### 5. Dataset and collate function

In [14]:
# ============================================================
# 5. DATASET + COLLATE
# ============================================================
class AudioDataset(Dataset):
    def __init__(self, cached_paths, labels):
        self.paths  = cached_paths
        self.labels = np.asarray(labels) if labels is not None else None

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        iv    = torch.from_numpy(np.load(self.paths[idx]))
        label = torch.tensor(float(self.labels[idx])) if self.labels is not None \
                else torch.tensor(-1.0)
        return iv, label


def collate_fn(batch):
    ivs, labels = zip(*batch)
    max_len = max(x.shape[0] for x in ivs)
    padded  = torch.zeros(len(ivs), max_len)
    for i, x in enumerate(ivs):
        padded[i, :x.shape[0]] = x
    return padded, torch.stack(labels)

### 6. SSL4PR model

In [15]:
# ============================================================
# 6. SSL4PR MODEL
# ============================================================
class AttentionPooling(nn.Module):
    """Collapses (B, T, D) → (B, D) with a learned query."""
    def __init__(self, dim: int):
        super().__init__()
        self.q = nn.Linear(dim, 1, bias=False)

    def forward(self, x):
        w = F.softmax(self.q(x).squeeze(-1), -1)
        return (w.unsqueeze(-1) * x).sum(1)


class SSL4PRModel(nn.Module):
    """
    backbone (output_hidden_states=True)
       → weighted sum across L+1 layers
       → temporal pooling
       → Dropout → Linear(1)

    By default the backbone is a deepcopy of a cached template, with the RNG
    state restored to where from_pretrained would have left it (see
    _restore_rng_after_backbone) so init is bit-identical to the original.
    Pass _legacy_from_pretrained=True to force a fresh from_pretrained load
    (used only by the startup bit-identity check).
    """
    def __init__(self, backbone_name=BACKBONE_NAME, backbone_type=BACKBONE_TYPE,
                 dropout=0.1, pooling="attention", freeze_feature_extractor=True,
                 random_state=BASE_SEED, _legacy_from_pretrained=False):
        super().__init__()
        self.pooling_type = pooling

        if _legacy_from_pretrained:
            self.backbone = _load_backbone(backbone_name, backbone_type)
        else:
            self.backbone = copy.deepcopy(_get_backbone_template(backbone_name, backbone_type))

        if freeze_feature_extractor:
            self.backbone.feature_extractor._freeze_parameters()

        if not _legacy_from_pretrained:
            _restore_rng_after_backbone(random_state, backbone_name, backbone_type)

        D = self.backbone.config.hidden_size
        L = self.backbone.config.num_hidden_layers + 1   # +1 for embedding layer

        self.layer_weights = nn.Parameter(torch.ones(L) / L)
        self.pool          = AttentionPooling(D) if pooling == "attention" else None
        self.drop          = nn.Dropout(dropout)
        self.head          = nn.Linear(D, 1)

    def forward(self, x):
        hs = torch.stack(
            self.backbone(x, output_hidden_states=True).hidden_states, dim=1
        )                                                  # (B, L, T, D)
        w  = F.softmax(self.layer_weights, 0)              # (L,)
        z  = (hs * w[None, :, None, None]).sum(1)          # (B, T, D)
        if self.pooling_type == "attention":
            z = self.pool(z)
        elif self.pooling_type == "mean":
            z = z.mean(1)
        else:
            z = z.max(1).values
        return self.head(self.drop(z))                     # (B, 1)

### 7. Scikit-learn-style wrapper

In [16]:
# ============================================================
# 7. SKLEARN-STYLE WRAPPER
# ============================================================
class SklearnSSL4PR:
    """fit / predict_proba / predict on lists of file paths."""
    def __init__(self, lr=1e-4, weight_decay=1e-5, batch_size=16,
                 dropout=0.1, pooling="attention",
                 freeze_feature_extractor=True,
                 backbone_name=BACKBONE_NAME, backbone_type=BACKBONE_TYPE,
                 random_state=BASE_SEED):
        self.lr                       = lr
        self.weight_decay             = weight_decay
        self.batch_size               = batch_size
        self.dropout                  = dropout
        self.pooling                  = pooling
        self.freeze_feature_extractor = freeze_feature_extractor
        self.backbone_name            = backbone_name
        self.backbone_type            = backbone_type
        self.random_state             = random_state
        self.best_epoch_              = 0
        self.best_val_auc_            = -1.0
        self.classes_                 = np.array([0, 1])

    def _loader(self, paths, labels, shuffle):
        ds = AudioDataset(paths, labels)
        gen = None
        if shuffle:
            gen = torch.Generator()
            gen.manual_seed(int(self.random_state))
        return DataLoader(
            ds, batch_size=self.batch_size, shuffle=shuffle,
            collate_fn=collate_fn, num_workers=0,
            pin_memory=DEVICE.type == "cuda",
            generator=gen,
        )

    def fit(self, paths, y, eval_set=None, patience=5, max_epochs=30, trial=None):
        set_seed(int(self.random_state))

        y = np.asarray(y)
        self.model = SSL4PRModel(
            backbone_name=self.backbone_name, backbone_type=self.backbone_type,
            dropout=self.dropout, pooling=self.pooling,
            freeze_feature_extractor=self.freeze_feature_extractor,
            random_state=self.random_state,
        ).to(DEVICE)

        opt   = torch.optim.AdamW(self.model.parameters(),
                                  lr=self.lr, weight_decay=self.weight_decay)
        crit  = nn.BCEWithLogitsLoss()
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and AMP_DTYPE is torch.float16)
        tr_dl = self._loader(paths, y, shuffle=True)

        has_val = eval_set is not None
        if has_val:
            vp, vy = eval_set
            val_dl = self._loader(vp, np.asarray(vy), shuffle=False)

        best_auc, best_w, no_imp = -1.0, None, 0
        self.best_epoch_ = max_epochs - 1

        for epoch in range(max_epochs):
            self.model.train()
            for bx, by in tr_dl:
                bx, by = bx.to(DEVICE), by.to(DEVICE).unsqueeze(1)
                opt.zero_grad()
                with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                    loss = crit(self.model(bx), by)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)                       # unscale before clipping
                nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                del bx, by, loss

            if has_val:
                vprobs, vy_np = self._infer(val_dl)
                auc = safe_auc(vy_np, vprobs)
                # Treat a NaN AUC (degenerate val fold) as "no improvement".
                if not np.isnan(auc) and auc > best_auc:
                    best_auc         = auc
                    best_w           = copy.deepcopy(self.model.state_dict())
                    self.best_epoch_ = epoch
                    no_imp           = 0
                else:
                    no_imp += 1

                if trial is not None:
                    trial.report(auc, epoch)
                    if trial.should_prune():
                        raise optuna.TrialPruned()

                print(f"    epoch {epoch+1:3d} | val_auc={auc:.4f} | "
                      f"best={best_auc:.4f} | no_imp={no_imp}", end="\r")

                if no_imp >= patience:
                    break

        if has_val and best_w is not None:
            self.model.load_state_dict(best_w)
            self.best_val_auc_ = best_auc

        return self

    @torch.no_grad()
    def _infer(self, loader):
        self.model.eval()
        probs, ys = [], []
        for bx, by in loader:
            bx = bx.to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = self.model(bx)
            p  = torch.sigmoid(logits).squeeze(1).float().cpu().numpy()
            probs.append(p)
            ys.append(by.numpy())
            del bx
        return np.concatenate(probs), np.concatenate(ys)

    def predict_proba(self, paths):
        dl   = self._loader(paths, np.zeros(len(paths)), shuffle=False)
        p, _ = self._infer(dl)
        return np.column_stack([1 - p, p])

    def predict(self, paths):
        return (self.predict_proba(paths)[:, 1] >= 0.5).astype(int)

### 8. Preprocess and cache audio

In [17]:
# ============================================================
# 8. PREPROCESS + CACHE
# ============================================================
print("\nPreprocessing and caching audio …")
fe           = AutoFeatureExtractor.from_pretrained(BACKBONE_NAME)
cached_paths = preprocess_and_cache(audio_paths, fe, MAX_AUDIO_SEC, TARGET_SR)

shapes = [np.load(p).shape for p in cached_paths[:20]]
assert len(set(shapes)) == 1, f"Inconsistent cached shapes: {set(shapes)}"
print(f"  Cache ready: {len(cached_paths)} files | shape: {shapes[0]}")


Preprocessing and caching audio …
  Cache ready: 1500 files | shape: (112000,)


### 8b. Verify that backbone caching is bit-identical

In [18]:
# ============================================================
# 8b. VERIFY BACKBONE CACHING IS BIT-IDENTICAL
# ============================================================
def verify_backbone_caching_identical(seed=BASE_SEED):
    """Build a model the ORIGINAL way (from_pretrained) and the NEW way
    (cached deepcopy + RNG replay) under the same seed and assert every
    parameter is bit-identical. Fails loudly if this environment's
    from_pretrained RNG behavior breaks the equivalence."""
    set_seed(seed)
    m_old = SSL4PRModel(random_state=seed, _legacy_from_pretrained=True)
    set_seed(seed)
    m_new = SSL4PRModel(random_state=seed)

    sd_o, sd_n = m_old.state_dict(), m_new.state_dict()
    assert sd_o.keys() == sd_n.keys(), "state_dict keys differ between paths"
    for k in sd_o:
        assert torch.equal(sd_o[k], sd_n[k]), \
            f"❌ init mismatch at '{k}' — backbone caching is NOT bit-identical"

    del m_old, m_new
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("✅ Backbone caching verified bit-identical to the from_pretrained path")

print("\nVerifying backbone-caching reproducibility …")
verify_backbone_caching_identical()

_tmp = SSL4PRModel(random_state=BASE_SEED)
tr, fr, tot = count_parameters(_tmp)
print(f"Trainable: {tr:,} | Frozen: {fr:,} | Total: {tot:,}")
del _tmp; gc.collect()


Verifying backbone-caching reproducibility …


Loading weights: 100%|██████████| 248/248 [00:00<00:00, 29566.74it/s]


✅ Backbone caching verified bit-identical to the from_pretrained path
Trainable: 90,183,038 | Frozen: 4,200,448 | Total: 94,383,486


0

### 9. Result storage

In [19]:
# ============================================================
# 9. RESULT STORAGE
# ============================================================
metrics_keys    = ["accuracy", "f1", "auc", "precision", "recall"]
results_sample  = {k: [] for k in metrics_keys}
results_speaker = {k: [] for k in metrics_keys}

# Per-fold (fold-to-fold) records, one row per (repeat, outer fold)
fold_records = []

### 10. Repeated nested cross-validation with Optuna (very slow)

In [ ]:
# ============================================================
# 10. NESTED CROSS-VALIDATION
# ============================================================
print(f"\nStarting SSL4PR Repeated Nested CV ({N_REPEATS} × {N_OUTER_SPLITS} folds)")

for repeat in range(N_REPEATS):
    repeat_col   = f"Repeat_{repeat+1}_Fold"
    current_seed = BASE_SEED + repeat
    set_seed(current_seed)

    print(f"\n{'='*80}")
    print(f"REPEAT {repeat+1}/{N_REPEATS}  (seed={current_seed} | col={repeat_col})")
    print("="*80)

    for fold in tqdm(range(N_OUTER_SPLITS), desc=f"Fold"):
        fold_seed = BASE_SEED + repeat * N_OUTER_SPLITS + fold

        test_mask  = (data[repeat_col] == fold).values
        train_mask = ~test_mask

        idx_tr = np.where(train_mask)[0]
        idx_te = np.where(test_mask)[0]

        paths_tr = [cached_paths[i] for i in idx_tr]
        paths_te = [cached_paths[i] for i in idx_te]
        y_tr, y_te = y[idx_tr], y[idx_te]
        g_tr, g_te = groups[idx_tr], groups[idx_te]
        ys_tr      = y_stratify[idx_tr]

        # LEAKAGE CHECK 1: Outer speaker overlap 
        outer_overlap = set(g_tr) & set(g_te)
        assert len(outer_overlap) == 0, \
            f"❌ OUTER LEAKAGE R{repeat+1} Fold {fold+1}: {outer_overlap}"

        # LEAKAGE CHECK 2: Speaker counts 
        n_spk_tr = len(set(g_tr))
        n_spk_te = len(set(g_te))
        assert n_spk_tr + n_spk_te == len(np.unique(groups)), \
            f"❌ Speaker count mismatch: {n_spk_tr}+{n_spk_te} != {len(np.unique(groups))}"

        # LEAKAGE CHECK 3: Sample index overlap 
        assert len(set(idx_tr) & set(idx_te)) == 0, \
            f"❌ SAMPLE OVERLAP R{repeat+1} Fold {fold+1}"
        assert len(paths_tr) + len(paths_te) == len(data), "Row count mismatch"

        # LEAKAGE CHECK 4: No shared cached inputs 
        assert len(set(paths_tr) & set(paths_te)) == 0, \
            f"❌ CACHE OVERLAP R{repeat+1} Fold {fold+1}"

        # LEAKAGE CHECK 5: Inner CV speaker overlap (pre-flight) 
        inner_cv_check = StratifiedGroupKFold(
            n_splits=N_INNER_SPLITS, shuffle=True,
            random_state=current_seed + fold,
        )
        for i, (tr_i, va_i) in enumerate(inner_cv_check.split(paths_tr, ys_tr, g_tr)):
            overlap_inner = set(g_tr[tr_i]) & set(g_tr[va_i])
            assert len(overlap_inner) == 0, \
                f"❌ INNER LEAKAGE R{repeat+1} Fold {fold+1} Inner {i}: {overlap_inner}"

        print(f"\nR{repeat+1} Fold {fold+1:2d}: ✅ All checks passed | "
              f"Train: {len(paths_tr)} samples ({n_spk_tr} spk) | "
              f"Test: {len(paths_te)} samples ({n_spk_te} spk)")

        print(f"    --> Starting Optuna Tuning (R{repeat+1} Fold {fold+1})...")
        # ────────────────────────────────────────────────────
        # INNER CV — Optuna hyperparameter search
        # ────────────────────────────────────────────────────
        def objective(trial):
            params = {
                "lr":           trial.suggest_float("lr", 1e-5, 1e-3, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
                "dropout":      trial.suggest_float("dropout", 0.0, 0.5),
                "pooling":      trial.suggest_categorical("pooling", ["attention", "mean", "max"]),
            }
            # batch_size is fixed at 16 (SklearnSSL4PR default) — no longer tuned.

            inner_cv = StratifiedGroupKFold(
                n_splits=N_INNER_SPLITS, shuffle=True,
                random_state=current_seed + fold,
            )

            auc_scores  = []
            best_epochs = []

            for inn_idx, (tr_idx, val_idx) in enumerate(
                inner_cv.split(paths_tr, ys_tr, g_tr)
            ):
                p_itr  = [paths_tr[i] for i in tr_idx]
                y_itr  = y_tr[tr_idx]
                p_ival = [paths_tr[i] for i in val_idx]
                y_ival = y_tr[val_idx]

                g_itr  = g_tr[tr_idx]
                g_ival = g_tr[val_idx]
                assert len(set(g_itr) & set(g_ival)) == 0, \
                    f"❌ INNER LEAKAGE inner fold {inn_idx}"

                m = SklearnSSL4PR(
                    **params,
                    backbone_name=BACKBONE_NAME, backbone_type=BACKBONE_TYPE,
                    random_state=fold_seed,
                )
                m.fit(p_itr, y_itr, eval_set=(p_ival, y_ival),
                      patience=5, max_epochs=30, trial=None)

                # Use best val AUC tracked during training — avoids double-eval
                auc_scores.append(m.best_val_auc_)
                best_epochs.append(m.best_epoch_ + 1)

                del m
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                trial.report(float(np.mean(auc_scores)), step=inn_idx)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            trial.set_user_attr(
                "optimal_epochs",
                max(1, int(np.round(np.mean(best_epochs))))
            )
            return float(np.mean(auc_scores))

        study = optuna.create_study(
            direction="maximize",
            sampler=TPESampler(seed=fold_seed),
            pruner=optuna.pruners.MedianPruner(),
        )
        study.optimize(objective, n_trials=N_OPTUNA_TRIALS, gc_after_trial=True)

        best_params    = study.best_params
        optimal_epochs = study.best_trial.user_attrs["optimal_epochs"]

        final = SklearnSSL4PR(
            **best_params,
            backbone_name=BACKBONE_NAME, backbone_type=BACKBONE_TYPE,
            random_state=fold_seed,
        )
        final.fit(paths_tr, y_tr, eval_set=None, max_epochs=optimal_epochs)

        # Evaluation
        y_pred  = final.predict(paths_te)
        y_proba = final.predict_proba(paths_te)[:, 1]

        results_sample["accuracy"].append( accuracy_score( y_te, y_pred))
        results_sample["f1"].append(       f1_score(       y_te, y_pred, zero_division=0))
        results_sample["auc"].append(      safe_auc(       y_te, y_proba))
        results_sample["precision"].append(precision_score(y_te, y_pred, zero_division=0))
        results_sample["recall"].append(   recall_score(   y_te, y_pred, zero_division=0))

        y_spk, p_spk, _ = aggregate_mean_by_group(y_te, y_proba, g_te)
        yp_spk          = (p_spk >= 0.5).astype(int)

        results_speaker["accuracy"].append( accuracy_score( y_spk, yp_spk))
        results_speaker["f1"].append(       f1_score(       y_spk, yp_spk, zero_division=0))
        results_speaker["auc"].append(      safe_auc(       y_spk, p_spk))
        results_speaker["precision"].append(precision_score(y_spk, yp_spk, zero_division=0))
        results_speaker["recall"].append(   recall_score(   y_spk, yp_spk, zero_division=0))

        fold_records.append({
            "repeat":           repeat + 1,
            "fold":             fold + 1,
            "fold_seed":        fold_seed,
            "best_lr":           best_params.get("lr"),
            "best_weight_decay": best_params.get("weight_decay"),
            "best_dropout":      best_params.get("dropout"),
            "best_pooling":      best_params.get("pooling"),
            "optimal_epochs":   optimal_epochs,
            "n_train_samples":  len(paths_tr),
            "n_test_samples":   len(paths_te),
            "n_train_speakers": n_spk_tr,
            "n_test_speakers":  n_spk_te,
            "sample_accuracy":  results_sample["accuracy"][-1],
            "sample_f1":        results_sample["f1"][-1],
            "sample_auc":       results_sample["auc"][-1],
            "sample_precision": results_sample["precision"][-1],
            "sample_recall":    results_sample["recall"][-1],
            "speaker_accuracy":  results_speaker["accuracy"][-1],
            "speaker_f1":        results_speaker["f1"][-1],
            "speaker_auc":       results_speaker["auc"][-1],
            "speaker_precision": results_speaker["precision"][-1],
            "speaker_recall":    results_speaker["recall"][-1],
        })

        print(f"           Best params: {best_params} | epochs={optimal_epochs}")
        print(f"           [Sample]  Acc={results_sample['accuracy'][-1]:.4f} | "
              f"AUC={results_sample['auc'][-1]:.4f} | F1={results_sample['f1'][-1]:.4f} | " f"Precision={results_sample['precision'][-1]:.4f} | "
              f"Recall={results_sample['recall'][-1]:.4f}")
        print(f"           [Speaker] Acc={results_speaker['accuracy'][-1]:.4f} | "
              f"AUC={results_speaker['auc'][-1]:.4f} | F1={results_speaker['f1'][-1]:.4f} | " f"Precision={results_speaker['precision'][-1]:.4f} | " f"Recall={results_speaker['recall'][-1]:.4f}")
        print("-"*70)
        

        del final, study
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

### 11. Save per-fold results

In [ ]:
# ============================================================
# 11. SAVE FOLD-TO-FOLD RESULTS
# ============================================================
fold_df = pd.DataFrame(fold_records)

fold_csv = f"fold_results_{RESULT_TAG}_{TASK}.csv"
fold_df.to_csv(fold_csv, index=False)
print(f"\nSaved per-fold results → {fold_csv}")

fold_txt = f"fold_results_{RESULT_TAG}_{TASK}.txt"
with open(fold_txt, "w", encoding="utf-8") as f:
    f.write(f"SSL4PR ({BACKBONE_NAME}) — PC-GITA {TASK}\n")
    f.write(f"Repeats x Folds: {N_REPEATS} x {N_OUTER_SPLITS} | Optuna trials: {N_OPTUNA_TRIALS}\n")
    f.write("=" * 110 + "\n")
    f.write(fold_df.to_string(index=False))
    f.write("\n")
print(f"Saved per-fold results → {fold_txt}")

### 12. Final results

In [ ]:
# ============================================================
# 12. FINAL RESULTS
# ============================================================
total_folds = N_REPEATS * N_OUTER_SPLITS
print(f"\n{'='*65}")
print(f"FINAL SSL4PR RESULTS  ({total_folds} total folds)")
print("="*65)
print(f"{'Metric':<12} | {'Sample-Level':<24} | {'Speaker-Level':<24}")
print("-"*65)

summary_rows = []
for k in metrics_keys:
    ms, ss = np.nanmean(results_sample[k]),  np.nanstd(results_sample[k])
    mk, sk = np.nanmean(results_speaker[k]), np.nanstd(results_speaker[k])
    print(f"{k.capitalize():<12} | {ms:.4f} ± {ss:.4f}          | {mk:.4f} ± {sk:.4f}")

for level, r in (("sample", results_sample), ("speaker", results_speaker)):
    row = {"level": level}
    for k in metrics_keys:
        row[f"{k}_mean"] = float(np.nanmean(r[k]))
        row[f"{k}_std"]  = float(np.nanstd(r[k]))
    summary_rows.append(row)
pd.DataFrame(summary_rows).to_csv(f"results_{RESULT_TAG}_{TASK}.csv", index=False)
print(f"\nSaved aggregated summary → results_{RESULT_TAG}_{TASK}.csv")

sys.stdout.log.close()
sys.stdout = sys.stdout.terminal